In [ ]:
!jq ".[$(jq 'keys[]' ../data/dataset.json | shuf -n 1)]" ../data/dataset.json

In [ ]:
!jq ".[$(jq 'keys[]' ../data/queries.json | shuf -n 1)]" ../data/queries.json

In [ ]:
import json 
from pathlib import Path 
from tinydb import TinyDB, Query
import re
import numpy as np
import arrowspace_tuner


db = TinyDB('db.json')
ROOT = Path.cwd().parent 
dataset_path = ROOT / "data" / "dataset.json"
query_path = ROOT / "data" / "nomic_embs" / "queries_emb_768.npy"
query_index_path = ROOT / "data" / "nomic_embs" / "queries_index.json"

DOC_PREFIX = "search_document: "

In [ ]:
def clean(text: str) -> str:
    """Remove {{placeholder}} tokens, collapse whitespace."""
    return re.sub(r"\s+", " ", re.sub(r"\{\{[^}]+\}\}", "", text)).strip()


def is_weak_title(el: dict) -> bool:
    """True when title has ≤4 words or shares no token with its own tags/category."""
    t = el["title"].lower()
    signals = el.get("tags", []) + [el.get("category", ""), el.get("subcategory", "")]
    has_signal = any(s.lower().replace("-", " ") in t for s in signals if s)
    return len(el["title"].split()) <= 4 or not has_signal


def extract_structured(el: dict) -> str:
    """
    Richest form: explicit field labels + full content.
    Named fields (Title:, Category:, Tags:) guide nomic attention to distribute
    semantic load evenly across ALL Matryoshka bands.
    Primary fix for Type-A failures (branding, coaching, communication, audit):
    engine is completely blind — needs every contextual signal available.
    """
    title  = el["title"].strip()
    tags   = ", ".join(el["tags"])
    body   = clean(el["content"])
    cat    = el["category"]
    subcat = el.get("subcategory", "")
    diff   = el.get("difficulty", "general")
    ph     = el.get("placeholders", [])
    ph_str = f"Variables: {', '.join(ph)}.\n" if ph else ""

    return (
        DOC_PREFIX
        + f"Title: {title}\n"
          f"Category: {cat} > {subcat}\n"
          f"Difficulty: {diff}\n"
          f"Tags: {tags}\n"
        + ph_str
        + body
    ).strip()


In [ ]:
with open(dataset_path, "r") as f:
    dataset = json.load(f)

In [ ]:
import numpy as np

def metadata_score(el: dict) -> float:
    likes     = np.log1p(el.get("likes", 0))
    upvotes   = np.log1p(el.get("upvotes", 0))
    downvotes = np.log1p(el.get("downvotes", 0))
    uses      = np.log1p(el.get("uses", 0))
    rep       = np.log1p(el.get("author_reputation", 0))
    forks     = np.log1p(el.get("fork_count", 0))
    vote_ratio = (upvotes + 1) / (upvotes + downvotes + 2)
    raw = (0.35 * vote_ratio + 0.20 * likes + 0.20 * rep
           + 0.10 * uses + 0.10 * forks)
    return float(raw)

meta_scores  = np.array([metadata_score(el) for el in dataset])
meta_norm = (meta_scores - meta_scores.min()) / ((meta_scores.max() - meta_scores.min()) + 1e-9)
print(f"meta_norm  min={meta_norm.min():.3f}  max={meta_norm.max():.3f}  mean={meta_norm.mean():.3f}")

In [ ]:
prompts = Query()

In [ ]:
for el in dataset:
    prompt = extract_structured(el)
    db.insert({'id' : el["id"], 'doc_string': prompt})

In [ ]:
db.search(prompts.id == "pk_03138")

### I ran this part on google colab due to gpu issues, I used the vs code extention for google colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install tinydb

In [ ]:
!ls /content/drive/MyDrive/prompt_kaban

In [ ]:
%cd /content/drive/MyDrive/prompt_kaban

In [ ]:
!uv run python nomic_corpus_embs.py \
  --dims 768 \
  --source-db db.json \
  --output-dir results

In [ ]:
!uv run python get_query_emb_nomic.py\
  --dims 768 \
  --output-dir results

## Load embeddings and build ArrowSpace

### Data Pipeline

TinyDB (db.json)

└── id + doc_string (20k records, sorted by id)

│
▼
nomic_corpus_embs.py ← encodes doc_strings, saves:

├── results/embeddings_nomic_structured_768d_raw.npy (N, 768) float32
├── results/embeddings_nomic_structured_768d_ids.npy (N,) str
└── results/embeddings_registry.json id → row_idx lookup
│
▼
ArrowSpaceBuilder.build(rows)
├── aspace ← spectral index with per-row λ (Rayleigh quotient)
└── gl ← GraphLaplacian (reused at query time)


Row alignment is guaranteed: `ids[i]` always corresponds to `embs[i]` and
`aspace.lambdas[i]`. The sort by id at save time makes this stable across runs.

> **⚠️ RAM budget (float64 in-memory)**
>
> | N items | 256d  | 512d  | 768d   |
> |---------|-------|-------|--------|
> | 20 k    | 40 MB | 80 MB | 120 MB |
> | 100 k   | 200 MB| 400 MB| 600 MB |
> | 500 k   | 1 GB  | 2 GB  | 3 GB   |
> | **1 M** | **2 GB** | **4 GB** | **6 GB** |
>
> ArrowSpace keeps a second copy of the matrix internally → **double the above**.
> For 1M prompts use **256d on a 16 GB machine**, or **768d on a 32 GB machine**.
> The Laplacian (sparse, k=6 neighbours) adds ~50 MB regardless of N.

In [ ]:
import numpy as np
from pathlib import Path

EMBS_DIR = ROOT / "data" / "nomic_embs" 
DIM      = 768  

emb_path = EMBS_DIR / f"embeddings_nomic_structured_{DIM}d_raw.npy"
ids_path = EMBS_DIR / f"embeddings_nomic_structured_{DIM}d_ids.npy"

# ArrowSpace requires float64
embs = np.load(emb_path).astype(np.float64)
ids  = list(np.load(ids_path))

# alignment guard — must never fail
assert embs.shape[0] == len(ids), \
    f"Shape mismatch: embs {embs.shape[0]} rows vs {len(ids)} ids"

print(f"Embeddings : {embs.shape}  dtype={embs.dtype}")
print(f"IDs        : {len(ids)} entries")
print(f"RAM in use : {embs.nbytes / 1e9:.2f} GB")
print(f"Sample     : ids[0]={ids[0]!r}  embs[0,:4]={embs[0, :4]}")

In [ ]:
from arrowspace_tuner import EpsTuner

tuner = EpsTuner(
    n_trials  = 15,
    sample_n  = 5_000,
    eps_low   = 0.5,
    eps_high  = 2.0,
    tau_low   = 0.42,
    tau_high  = 1.0,
    n_probe   = 50,
    storage   = "sqlite:///arrowspace_tuning.db",
)

aspace, gl = tuner.fit(embs)

In [ ]:
tuner.save_report()

In [ ]:
from arrowspace import ArrowSpaceBuilder

graph_params = {'eps' : 1.2310568997881801, 'k' : 38, 'p' : 2.0, 'sigma' : None}

aspace, gl = ArrowSpaceBuilder().build(graph_params, embs)

In [ ]:
import numpy as np
from arrowspace import ArrowSpaceBuilder

# ── 1. Build score lookup keyed by id ────────────────────────────────────────
meta_by_id = {el["id"]: el for el in dataset}

# ── 2. Align meta_norm to the SAME row order as embs (= ids from .npy) ───────
#    ids is sorted (pk_00001…pk_20000), dataset is NOT — alignment is required.
meta_scores_aligned = np.array(
    [metadata_score(meta_by_id[pk]) for pk in ids],   # ids = list(np.load(ids_path))
    dtype=np.float64
)
meta_norm_aligned = (
    (meta_scores_aligned - meta_scores_aligned.min())
    / (meta_scores_aligned.max() - meta_scores_aligned.min() + 1e-9)
)

# ── 3. Scale embeddings — floor=0.85 so zero-score prompts keep 85% signal ──
FLOOR = 0.85
scale = FLOOR + (1.0 - FLOOR) * meta_norm_aligned      # shape (20000,)  ∈ [0.85, 1.0]
embs_meta = embs * scale[:, np.newaxis]                 # broadcast over 768 dims

print(f"scale  min={scale.min():.4f}  max={scale.max():.4f}  mean={scale.mean():.4f}")
print(f"embs_meta shape: {embs_meta.shape}  dtype={embs_meta.dtype}")

# ── 4. Build ArrowSpace with meta-augmented embeddings ───────────────────────
graph_params = {"eps": 1.2310568997881801, "k": 38, "p": 2.0, "sigma": None}

print("\nBuilding aspace_base  (semantic only)…")
aspace_base, gl_base = ArrowSpaceBuilder().build(graph_params, embs)

print("Building aspace_meta  (semantic + engagement wiring)…")
aspace_meta, gl_meta = ArrowSpaceBuilder().build(graph_params, embs_meta)

lambdas_base = aspace_base.lambdas()
lambdas_meta = aspace_meta.lambdas()

print(f"\naspace_base  λ  mean={sum(lambdas_base)/len(lambdas_base):.6f}")
print(f"aspace_meta  λ  mean={sum(lambdas_meta)/len(lambdas_meta):.6f}")
print("\nBoth indexes ready — use aspace_meta/gl_meta for metadata-boosted retrieval.")

In [ ]:
# ── aspace_base stats ────────────────────────────────────────────────────────
lambdas_base = aspace_base.lambdas()
print("aspace_base (semantic only)")
print(f"  λ  min={min(lambdas_base):.6f}  max={max(lambdas_base):.6f}  "
      f"mean={sum(lambdas_base)/len(lambdas_base):.6f}")

# ── aspace_meta stats ────────────────────────────────────────────────────────
lambdas_meta = aspace_meta.lambdas()
print("\naspace_meta (semantic + engagement wiring)")
print(f"  λ  min={min(lambdas_meta):.6f}  max={max(lambdas_meta):.6f}  "
      f"mean={sum(lambdas_meta)/len(lambdas_meta):.6f}")

# ── default alias for downstream cells ───────────────────────────────────────
aspace, gl, lambdas = aspace_meta, gl_meta, lambdas_meta

In [ ]:
import numpy as np
import plotly.graph_objects as go
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from sklearn.decomposition import PCA
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter

# ── point to the meta-augmented embeddings & Laplacian ────────────────
rows = embs_meta   # swap to embs_base for the semantic-only manifold

# ── Helpers ────────────────────────────────────────────────────────────
def gl_to_scipy(gl) -> sp.csr_matrix:
    raw = gl.to_csr()
    return sp.csr_matrix(
        (np.asarray(raw[0]), np.asarray(raw[1]), np.asarray(raw[2])),
        shape=gl.shape()
    )

L       = gl_to_scipy(gl)   # gl = gl_meta alias set in the build cell
n_nodes = L.shape[0]
degrees = np.array(L.diagonal(), dtype=np.float64)




# ── Clip outliers ──────────────────────────────────────────────────────
p95  = np.percentile(degrees, 95)
p05  = np.percentile(degrees, 5)
z_pts = np.clip(degrees, p05, p95)


# ── 2D layout: rows (PCA) if available, else spectral embedding ────────
try:
    # 'rows' must be a (N, D) array-like already in scope
    if 'rows' not in dir() or rows is None:
        raise NameError("rows not defined")
    embs_2d  = np.array(rows[:n_nodes], dtype=np.float32)
    pca      = PCA(n_components=2)
    xy       = pca.fit_transform(embs_2d)
    x_pts, y_pts = xy[:, 0], xy[:, 1]
    var          = pca.explained_variance_ratio_
    x_label      = f'PC1 ({var[0]*100:.1f}%)'
    y_label      = f'PC2 ({var[1]*100:.1f}%)'
    subtitle     = f'PC1 {var[0]*100:.1f}%  PC2 {var[1]*100:.1f}%'

except NameError:
    # ── Fallback: Fiedler spectral embedding from L itself ────────────
    # Use the 2nd & 3rd smallest eigenvectors (skip trivial constant vec)
    k_eig = min(4, n_nodes - 1)          # need at least 3 eigenvectors
    vals, vecs = eigsh(
        L.astype(np.float64), k=k_eig, which='SM', tol=1e-5, maxiter=3000
    )
    order     = np.argsort(vals)         # sort ascending by eigenvalue
    vecs      = vecs[:, order]
    x_pts     = vecs[:, 1]              # Fiedler vector  (λ₂)
    y_pts     = vecs[:, 2]              # next eigenvector (λ₃)
    var       = None
    x_label   = f'Fiedler v (λ₂={vals[order[1]]:.3f})'
    y_label   = f'Spectral v (λ₃={vals[order[2]]:.3f})'
    subtitle  = 'spectral embedding  ·  Fiedler (λ₂) × λ₃'


# ── Interpolation on a fine grid ───────────────────────────────────────
grid_res = 120
xi = np.linspace(x_pts.min(), x_pts.max(), grid_res)
yi = np.linspace(y_pts.min(), y_pts.max(), grid_res)
Xi, Yi = np.meshgrid(xi, yi)

Zi      = griddata((x_pts, y_pts), z_pts, (Xi, Yi), method='cubic')
Zi_near = griddata((x_pts, y_pts), z_pts, (Xi, Yi), method='nearest')
Zi[np.isnan(Zi)] = Zi_near[np.isnan(Zi)]
Zi = gaussian_filter(Zi, sigma=2.5)


# ── Colorscale ────────────────────────────────────────────────────────
colorscale = [
    [0.0,  '#1a3a6b'],
    [0.25, '#4a90d9'],
    [0.5,  '#f5f5f5'],
    [0.75, '#e05c3a'],
    [1.0,  '#8b0000'],
]


# ── Surface ───────────────────────────────────────────────────────────
surface = go.Surface(
    x=Xi, y=Yi, z=Zi,
    colorscale=colorscale,
    opacity=1.0,
    showscale=True,
    colorbar=dict(
        title=dict(text='Degree (Curvature)', font=dict(size=12)),
        thickness=16, x=1.01, tickfont=dict(size=10),
    ),
    contours=dict(
        z=dict(
            show=True,
            usecolormap=True,
            highlightcolor='rgba(255,255,255,0.6)',
            project_z=True,
            start=float(Zi.min()),
            end=float(Zi.max()),
            size=(Zi.max() - Zi.min()) / 12,
        )
    ),
    lighting=dict(ambient=0.7, diffuse=0.9, specular=0.2,
                  roughness=0.6, fresnel=0.1),
    lightposition=dict(x=1000, y=1000, z=2000),
    hoverinfo='skip',
    name='Manifold',
)


# ── Hub-node scatter (top 15 % by degree) ─────────────────────────────
top_idx = np.where(degrees > np.percentile(degrees, 85))[0]
node_scatter = go.Scatter3d(
    x=x_pts[top_idx], y=y_pts[top_idx], z=z_pts[top_idx] + 1,
    mode='markers',
    marker=dict(
        size=5,
        color=degrees[top_idx],
        colorscale=colorscale,
        cmin=p05, cmax=p95,
        opacity=1.0,
        line=dict(width=0.8, color='white'),
    ),
    text=[f'Node {i}<br>degree={degrees[i]:.2f}' for i in top_idx],
    hoverinfo='text',
    name='Hub nodes (top 15%)',
)


# ── Figure ─────────────────────────────────────────────────────────────
fig = go.Figure(
    data=[surface, node_scatter],
    layout=go.Layout(
        title=dict(
            text=(
                f'Graph Laplacian Manifold  ·  {n_nodes} nodes<br>'
                f'<sup>Z = degree (local curvature, clip p5–p95)  ·  {subtitle}</sup>'
            ),
            x=0.5, font=dict(size=13, color='#e2e6f0'),
        ),
        scene=dict(
            xaxis=dict(title=x_label,
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            yaxis=dict(title=y_label,
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            zaxis=dict(title='Degree L_ii',
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            bgcolor='rgb(10,10,20)',
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.9),
                        up=dict(x=0, y=0, z=1)),
            aspectmode='manual',
            aspectratio=dict(x=1.6, y=1.6, z=0.55),
        ),
        paper_bgcolor='rgb(12,12,22)',
        font=dict(color='#e2e6f0', family='system-ui, sans-serif'),
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(0,0,0,0.5)',
                    font=dict(size=11)),
        margin=dict(l=0, r=0, t=90, b=0),
        height=700,
    )
)

fig.show(renderer="vscode")

# 🌐 Graph Laplacian Manifold — What You're Looking At

This surface is a **topographic map of your semantic space**.

Each point on the XY plane is a **centroid** (a cluster of embeddings from your dataset),
projected onto its first two principal components. The **height Z** is the diagonal of the
Graph Laplacian — the *degree* of that centroid in the ε-proximity graph built by ArrowSpace.

---

## 🔴 Red peaks = Dense semantic regions
High-degree centroids are **hubs**: many neighbours, tightly connected.
These are the dominant themes in your dataset — concepts the model understands well
and where retrieval is reliable.

## 🔵 Blue valleys = Sparse / frontier regions
Low-degree centroids are **outliers or boundary concepts**: few neighbours, weakly connected.
These are the *tail items* — the zone where standard cosine similarity loses precision
because there is not enough local structure to anchor results.

---

## Why this matters for search

A classical vector database treats the entire space as geometrically flat.
Every embedding is compared to a query with the same cosine formula, regardless
of whether it lives in a dense hub or an isolated valley.

**ArrowSpace does not.**

It builds this Laplacian, computes a per-item spectral score λ (the Rayleigh quotient),
and blends it with cosine similarity at query time:
score(q, item) = α · cosine(q, item) + (1-α) · λ_spectral(item)

text

Items in the valleys are re-weighted upward when they are genuinely relevant —
recovering the tail retrieval that flat cosine misses.

---

## What the shape of this surface tells you about your dataset

| Feature | Interpretation |
|---|---|
| Many tall red peaks | Strong thematic clusters — good dataset coverage |
| Very deep isolated blue holes | Potential coverage gaps or noisy embeddings |
| Smooth, gently rolling surface | Well-distributed, semantically coherent dataset |
| Sharp spikes (before clipping) | Outlier centroids — candidates for data cleaning |
| Surface changes over time | **Semantic drift** — your data distribution is shifting |

---

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GRAPH LAPLACIAN — DATASET FINGERPRINT ANALYSIS
# ════════════════════════════════════════════════════════════════════════════
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

# ── 1. Basic structural metrics ───────────────────────────────────────────
n          = L.shape[0]
nnz        = L.nnz
n_edges    = (nnz - n) // 2          # off-diagonal non-zeros / 2
sparsity   = 1.0 - nnz / (n * n)
avg_degree = float(degrees.mean())
std_degree = float(degrees.std())
max_degree = float(degrees.max())
min_degree = float(degrees.min())
degree_cv  = std_degree / avg_degree  # coefficient of variation: 0=flat, >1=spiky

# ── 2. Fiedler value (algebraic connectivity) ─────────────────────────────
# λ₁=0 always (constant vector); λ₂=Fiedler: how well-connected the graph is
try:
    diag_d     = np.array(L.diagonal(), dtype=np.float64)
    safe_d     = np.where(diag_d > 1e-12, diag_d, 1e-12)
    d_inv_sqrt = sp.diags(1.0 / np.sqrt(safe_d))
    L_norm     = d_inv_sqrt @ L @ d_inv_sqrt
    eigs       = spla.eigsh(L_norm, k=6, which="SM",
                            return_eigenvectors=False, tol=1e-5, maxiter=3000)
    eigs_sorted   = sorted(np.real(eigs))
    fiedler       = max(0.0, eigs_sorted[1])
    spectral_gap  = eigs_sorted[2] - eigs_sorted[1] if len(eigs_sorted) > 2 else 0.0
except Exception as e:
    fiedler, spectral_gap = 0.0, 0.0
    print(f"[warn] Fiedler computation failed: {e}")

# ── 3. Degree distribution buckets ────────────────────────────────────────
p10, p25, p50, p75, p90 = np.percentile(degrees, [10, 25, 50, 75, 90])
hub_count      = int((degrees > p90).sum())            # top 10% = hubs
isolated_count = int((degrees < p10).sum())            # bottom 10% = tails
hub_fraction   = hub_count / n * 100
tail_fraction  = isolated_count / n * 100

# ── 4. Connectivity interpretation ────────────────────────────────────────
def interpret_fiedler(f):
    if f < 0.01:  return "⚠️  Near-disconnected (multiple isolated clusters)"
    if f < 0.05:  return "🟡 Weakly connected (sparse bridge structure)"
    if f < 0.15:  return "🟢 Moderately connected (balanced topology)"
    return               "🔵 Strongly connected (dense, cohesive manifold)"

def interpret_cv(cv):
    if cv < 0.3:  return "Flat (uniform density, no dominant hubs)"
    if cv < 0.7:  return "Moderate variation (some hubs, mostly uniform)"
    if cv < 1.2:  return "High variation (clear hub/tail structure)"
    return               "Power-law-like (few mega-hubs, many isolated tails)"

def interpret_gap(gap):
    if gap < 0.02: return "Continuous spectrum (smooth manifold)"
    if gap < 0.1:  return "Soft cluster boundary (overlapping topics)"
    return                "Hard cluster boundary (distinct topic groups)"

# ── 5. Prepare Table Data ─────────────────────────────────────────────────
rows_table = [
    ("Nodes (centroids)",        f"{n}",                              "Clusters extracted from your embeddings"),
    ("Edges (connections)",      f"{n_edges:,}",                      "Semantic proximity links between centroids"),
    ("Sparsity",                 f"{sparsity*100:.2f}%",              "Most centroids are NOT directly connected"),
    ("Avg degree",               f"{avg_degree:.3f}",                 "Mean connectivity per centroid"),
    ("Degree std",               f"{std_degree:.3f}",                 "Spread of connectivity across centroids"),
    ("Degree CV",                f"{degree_cv:.3f}",                  interpret_cv(degree_cv)),
    ("Max degree (hub)",         f"{max_degree:.3f} (node {int(np.argmax(degrees))})", "Most connected centroid"),
    ("Min degree (tail)",        f"{min_degree:.3f} (node {int(np.argmin(degrees))})", "Most isolated centroid"),
    ("Hub centroids (top 10%)",  f"{hub_count} ({hub_fraction:.1f}% of nodes)",       "Dominant theme anchors"),
    ("Tail centroids (bot 10%)", f"{isolated_count} ({tail_fraction:.1f}% of nodes)", "Sparse / hard-to-retrieve zones"),
    ("Fiedler value λ₂",         f"{fiedler:.5f}",                    interpret_fiedler(fiedler)),
    ("Spectral gap λ₃−λ₂",       f"{spectral_gap:.5f}",               interpret_gap(spectral_gap)),
] 

# ── 6. Print Plain Text Table ─────────────────────────────────────────────
print("\n" + "═" * 105)
print(" 📊 GRAPH LAPLACIAN — DATASET FINGERPRINT")
print("═" * 105)
print(f" {'METRIC':<25} │ {'VALUE':<20} │ {'INTERPRETATION'}")
print("─" * 105)
for row in rows_table:
    print(f" {row[0]:<25} │ {row[1]:<20} │ {row[2]}")
print("═" * 105)

# ── 7. One-paragraph narrative ────────────────────────────────────────────
print("\n── NARRATIVE SUMMARY ──────────────────────────────────────────────────────────")
print(f"""
Dataset contains {n} semantic centroids wired into a graph with {n_edges:,} edges 
({sparsity*100:.1f}% sparse).

Degree distribution: avg={avg_degree:.2f} ± {std_degree:.2f} (CV={degree_cv:.2f})
→ {interpret_cv(degree_cv)}

{hub_count} hub centroids ({hub_fraction:.1f}%) anchor the dominant themes.
{isolated_count} tail centroids ({tail_fraction:.1f}%) are the hard-to-retrieve zone —
exactly where ArrowSpace's spectral re-weighting adds value over flat cosine.

Algebraic connectivity (Fiedler λ₂ = {fiedler:.5f}):
→ {interpret_fiedler(fiedler)}

Spectral gap (λ₃ − λ₂ = {spectral_gap:.5f}):
→ {interpret_gap(spectral_gap)}
""")

# Graph Laplacian Fingerprint — What the Numbers Say

---

## The Dataset at a Glance

| | |
|---|---|
| **768 centroids** | semantic clusters automatically extracted from your embeddings |
| **4,791 edges** | proximity links — the wiring of your semantic space |
| **98.25% sparse** | the graph is lean: each centroid connects to ~7 neighbours on average |

---

## The Core Finding: Your Data Has a Power-Law Structure

The degree distribution has a **Coefficient of Variation of 4.13** — well into power-law territory.

This means your semantic space looks like this:
Degree
│ 266 ┤ █ ← node 69: mega-hub (one dominant concept cluster)

│ █

│ █ █

│ █ █ █

│ █ █ █ █ █

0 ┤─────────────────────────────────────► Centroids
few hubs long tail of isolated concepts

text

**A handful of concepts absorb most of the connections.  
Hundreds of concepts sit in near-isolation.**

This is not a data quality problem. It is the natural geometry of language —
the same power-law Zipf observed in word frequencies in 1935.

---

## Why This Is Exactly the Problem ArrowSpace Solves

Standard cosine similarity is **blind to this structure**.
It assigns the same retrieval logic to node 69 (degree 266) and node 51 (degree 0).

The result is predictable:

| Zone | What cosine does | What ArrowSpace does |
|---|---|---|
| 🔴 **Hub centroids** (top 10%, degree ≫ avg) | Works well — many neighbours reinforce the result | Works well + detects over-representation bias |
| 🔵 **Tail centroids** (bottom 10%, degree ≈ 0) | Retrieval fails silently — no neighbours to anchor results | Spectral re-weighting surfaces these items when genuinely relevant |


> **The tail is not noise. It is your users' hardest, most specific queries —
> the ones that matter most and fail most often.**

---

## The Graph Is Globally Healthy

**Fiedler value λ₂ = 0.649**

The Fiedler value measures algebraic connectivity — how hard it is to
"cut" the graph into disconnected pieces.

- Typical real-world graphs: **0.01 – 0.10**
- Your dataset: **0.649** → exceptionally well-connected

This tells you two things:

1. The dataset is **one coherent semantic territory**, not a fragmented archipelago
2. The ArrowSpace graph parameters (`eps=1.027, k=20`) are **well-calibrated** —
   the graph is neither over-wired nor broken

---

## The Manifold Is Continuous, Not Clustered

**Spectral gap λ₃ − λ₂ = 0.016**

A large spectral gap would indicate hard, well-separated topic clusters.
A near-zero gap (like yours) means the topics **blend gradually into each other** —
a smooth semantic manifold, not discrete buckets.

This has a direct consequence for retrieval:

> Hard clustering (k-means, topic models) **destroys information** on this dataset.  
> You need a method that respects the continuous geometry.  
> That is what ArrowSpace's taumode scoring does.

---

## The One Node Worth Inspecting

**Node 51 — degree = 0.000**

This centroid has zero connections in the proximity graph.
It represents a prompt (or group of prompts) that is **semantically unique** —
nothing in the dataset is similar enough to reach it under the current `eps`.

With flat cosine similarity: this item is **effectively invisible** to the retrieval system.  
With ArrowSpace: it can be **flagged, boosted, or used as a coverage gap signal**.

---

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LAMBDA DISTRIBUTION — SPECTRAL FINGERPRINT  (base vs meta)
# ════════════════════════════════════════════════════════════════════════════
import numpy as np
import scipy.sparse as sp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── pull lambdas from both indexes ───────────────────────────────────────────
lambdas_base_all = np.array(aspace_base.lambdas(), dtype=np.float64)
lambdas_meta_all = np.array(aspace_meta.lambdas(), dtype=np.float64)   # = lambdas alias

def _stats(lam):
    p10, p25, p50, p75, p90, p95 = np.percentile(lam, [10, 25, 50, 75, 90, 95])
    return dict(
        min=lam.min(), max=lam.max(), mean=lam.mean(),
        median=p50, std=lam.std(),
        cv=lam.std()/lam.mean() if lam.mean()>0 else 0,
        p10=p10, p25=p25, p50=p50, p75=p75, p90=p90, p95=p95,
        smooth=int((lam <= p25).sum()),
        mid=int(((lam > p25) & (lam <= p75)).sum()),
        rough=int((lam > p75).sum()),
        n=len(lam),
    )

sb = _stats(lambdas_base_all)
sm = _stats(lambdas_meta_all)

# ── terminal printout ─────────────────────────────────────────────────────────
for label, s in [("BASE (semantic only)", sb), ("META (+ engagement wiring)", sm)]:
    print("\n" + "═"*80)
    print(f" SPECTRAL λ — {label}")
    print("═"*80)
    print(f"   Mean / Median     : {s['mean']:.6f}  /  {s['median']:.6f}")
    print(f"   Std / CV          : {s['std']:.6f}  /  {s['cv']:.3f}")
    print(f"   Min / Max         : {s['min']:.6f}  /  {s['max']:.6f}")
    print(f"   p25 / p75         : {s['p25']:.6f}  /  {s['p75']:.6f}")
    print(f"   Smooth / Mid / Rough : {s['smooth']:,}  /  {s['mid']:,}  /  {s['rough']:,}")
print("═"*80)

# ── shared x-range for both plots ─────────────────────────────────────────────
x_max = max(sb['p95'], sm['p95'])
bins  = np.linspace(0, x_max, 300)

def _hist(lam, bins):
    h, edges = np.histogram(lam, bins=bins)
    return (edges[:-1] + edges[1:]) / 2, h

bc_base, h_base = _hist(lambdas_base_all, bins)
bc_meta, h_meta = _hist(lambdas_meta_all, bins)

# ── figure: 2 rows × 1 col ────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "λ Histogram — Base vs Meta (zoomed to p95)",
        "ECDF — Base vs Meta"
    ),
    vertical_spacing=0.14,
)

# ── row 1: overlaid histograms ────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=bc_base, y=h_base,
    name='Base (semantic)',
    marker_color='rgba(74,144,217,0.55)',
    marker_line_width=0,
    hovertemplate='λ=%{x:.4f}  count=%{y}<extra>base</extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=bc_meta, y=h_meta,
    name='Meta (+ salience)',
    marker_color='rgba(253,171,67,0.55)',
    marker_line_width=0,
    hovertemplate='λ=%{x:.4f}  count=%{y}<extra>meta</extra>',
), row=1, col=1)

# mean lines
for val, color, label in [
    (sb['mean'], '#4a90d9', f"base mean={sb['mean']:.4f}"),
    (sm['mean'], '#fdab43', f"meta mean={sm['mean']:.4f}"),
]:
    fig.add_vline(x=val, line_dash='dash', line_color=color,
                  line_width=1.5, row=1, col=1)
    fig.add_annotation(
        x=val, y=0.92, yref="y domain", row=1, col=1,
        text=label, showarrow=False, xanchor="left", xshift=6,
        font=dict(color=color, size=10)
    )

# ── row 2: ECDF overlay ───────────────────────────────────────────────────────
for lam, color, name in [
    (lambdas_base_all, '#4a90d9', 'Base (semantic)'),
    (lambdas_meta_all, '#fdab43', 'Meta (+ salience)'),
]:
    sl = np.sort(lam)
    ey = np.arange(1, len(sl)+1) / len(sl)
    fig.add_trace(go.Scatter(
        x=sl, y=ey, mode='lines',
        line=dict(color=color, width=2),
        name=name,
        hovertemplate='λ=%{x:.4f}  pct=%{y:.2%}<extra>'+name+'</extra>',
    ), row=2, col=1)

# zone shading on ECDF (use meta thresholds as reference)
for x0, x1, col, label in [
    (0,          sm['p25'], 'rgba(74,144,217,0.10)',  'Smooth'),
    (sm['p25'],  sm['p75'], 'rgba(200,200,200,0.07)', 'Mid'),
    (sm['p75'],  x_max,     'rgba(192,57,43,0.10)',   'Rough'),
]:
    fig.add_vrect(x0=x0, x1=x1, fillcolor=col, layer='below',
                  line_width=0, row=2, col=1,
                  annotation_text=label, annotation_position='top left',
                  annotation_font_size=10, annotation_font_color='#aaa')

# ── layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    barmode='overlay',
    title=dict(
        text=(
            f'Spectral λ Distribution  ·  {sm["n"]:,} items  ·  Base vs Meta<br>'
            f'<sup>'
            f'base  mean={sb["mean"]:.4f}  CV={sb["cv"]:.3f}  '
            f'rough={sb["rough"]} ({sb["rough"]/sb["n"]*100:.0f}%)  │  '
            f'meta  mean={sm["mean"]:.4f}  CV={sm["cv"]:.3f}  '
            f'rough={sm["rough"]} ({sm["rough"]/sm["n"]*100:.0f}%)'
            f'</sup>'
        ),
        x=0.5, font=dict(size=13, color='#e2e6f0'),
    ),
    paper_bgcolor='rgb(12,12,22)',
    plot_bgcolor='rgb(18,18,32)',
    font=dict(color='#e2e6f0', family='system-ui, sans-serif'),
    legend=dict(x=0.75, y=0.97, bgcolor='rgba(0,0,0,0.5)', font=dict(size=11)),
    height=820,
    margin=dict(l=40, r=40, t=110, b=40),
)

fig.update_xaxes(title_text='λ (Rayleigh quotient)', range=[0, x_max],
                 gridcolor='rgba(100,100,160,0.2)', row=1, col=1)
fig.update_xaxes(title_text='λ (Rayleigh quotient)',
                 gridcolor='rgba(100,100,160,0.2)', row=2, col=1)
fig.update_yaxes(title_text='Count',
                 range=[0, max(h_base[1:].max(), h_meta[1:].max()) * 1.8],
                 gridcolor='rgba(100,100,160,0.2)', row=1, col=1)
fig.update_yaxes(title_text='Percentile', tickformat='.0%',
                 gridcolor='rgba(100,100,160,0.2)', row=2, col=1)

fig.show(renderer="vscode")

# ── narrative linking λ shift to metadata wiring ──────────────────────────────
delta_mean  = sm['mean']  - sb['mean']
delta_rough = sm['rough'] - sb['rough']
cv_interp   = ("Smoother than degree: spectral smoothing is working."
               if sm['cv'] <= 1.5 else "Matches degree: λ also power-law distributed.")

print(f"""
 ── BASE vs META λ SHIFT ──────────────────────────────────────────────────────

 Mean λ shift      : {sb['mean']:.4f} → {sm['mean']:.4f}  (Δ={delta_mean:+.4f})
 Rough-zone shift  : {sb['rough']:,} → {sm['rough']:,}  (Δ={delta_rough:+,} items)
 CV shift          : {sb['cv']:.3f} → {sm['cv']:.3f}

 A positive Δ mean means high-salience prompts pulled more edges into the graph,
 raising the average Rayleigh quotient — exactly the intended topology change.
 The rough-zone shift shows how many prompts moved out of the tail into the
 mid-range, making them more retrievable at query time via taumode scoring.

 Graph degree CV   : {degree_cv:.3f}  → Power-law degree distribution
 Lambda CV (meta)  : {sm['cv']:.3f}  → {cv_interp}

 [ SMOOTH ]  {sm['smooth']:,} items (λ ≤ {sm['p25']:.4f}) — near hub centroids, easy to retrieve
 [ MID    ]  {sm['mid']:,} items — standard structural zone
 [ ROUGH  ]  {sm['rough']:,} items (λ > {sm['p75']:.4f}) — spectrally distinctive, ArrowSpace gains here
 ──────────────────────────────────────────────────────────────────────────────
""")

# 📈 Spectral λ Distribution — Reading the Shape of Your Data

---

## What the Plots Reveal

The two charts together tell a coherent story that links directly to the Laplacian fingerprint.

### Histogram — a bimodal-like shape with a flat plateau

The histogram shows a **sharp spike near λ ≈ 0** followed by a remarkably flat
plateau from λ ≈ 0.005 to λ ≈ 0.21, then a gradual red tail extending to λ = 1.0.

This is **not** a bell curve. It has three distinct regimes:
Count
│ ██ ← spike: 5,000 smooth items crammed near λ = 0

│ (hub-adjacent — cosine works fine here)

████████████████████████████ ← flat plateau: 10,000 mid items

│ (uniform structural density)

│ █████████████████████████████████ ← red tail: 5,000 rough items

│ (spectrally isolated — cosine misses these)
└──────────────────────────────────────────────► λ
0 0.05 0.21 1.0

text

The flat plateau is a key insight: mid-range items are **uniformly distributed**
across the spectral axis. This means there is no natural hard boundary between
"easy" and "hard" retrieval — the difficulty is a continuum.

---

### ECDF — the steepness tells the whole story

The ECDF rises steeply from 0 → 80% in the narrow range λ ∈ [0, 0.21],
then flattens into a long shallow tail from λ ∈ [0.21, 1.0].

| ECDF Region | λ Range | % Items | Interpretation |
|---|---|---|---|
| Steep rise | 0 → 0.21 | ~75% | Dense, hub-adjacent — fast to retrieve |
| Flat tail | 0.21 → 1.0 | ~25% | Sparse, isolated — where retrieval fails |

The **"elbow" at λ ≈ 0.21** is the natural operating boundary of cosine similarity.
Everything to the right of that elbow is the **ArrowSpace opportunity zone**.

---

## Linking Back to the Laplacian Fingerprint

| Laplacian Metric | Value | λ Distribution Echo |
|---|---|---|
| Degree CV | 4.131 (power-law) | λ CV = 1.311 — smoother, but still skewed |
| Fiedler λ₂ = 0.649 | Strong global cohesion | No gap in ECDF → no hard cluster boundary |
| Spectral gap = 0.016 | Continuous manifold | Flat histogram plateau confirms smooth topology |
| Hub centroids (10%) | 77 mega-hubs | The λ ≈ 0 spike — 5,000 items anchored to hubs |
| Tail centroids (10%) | 77 isolated nodes | The red ECDF tail — 5,000 items with λ > 0.21 |

> **Key insight:** The degree CV (4.131) is 3× higher than the λ CV (1.311).
> This compression is spectral smoothing in action — the Laplacian integrates
> neighbourhood structure and dampens the raw degree extremes into a more
> continuous signal. ArrowSpace scores operate on this smoothed geometry,
> not on the raw noisy degree counts.

---

## The Retrieval Gap — Quantified
Total items : 20,000
────────────────────────────────────────────────────
Cosine covers well: 15,000 items (smooth + mid)
Cosine misses: 5,000 items (rough zone, λ > 0.215)
──────
25% of your dataset is invisible to standard search

text

Those 5,000 rough items are not low-quality data.
They are your **most specific, most unique, hardest-to-match prompts** —
exactly the ones power users query, and exactly the ones that return
poor results today.

ArrowSpace's taumode blends the Rayleigh quotient λ into the similarity score
at query time, pulling rough items up in the ranking when they are
genuinely the best match — without penalising smooth items.

---

In [ ]:
dataset[0]

In [ ]:

q_embs = np.load(query_path).astype(np.float64)
idx    = json.load(open(query_index_path))

row    = idx["q_09"]["row_index"]          # 8
query  = q_embs[row].astype(np.float64)

In [ ]:
ids_path = ROOT / "data" / "nomic_embs" / "embeddings_nomic_structured_768d_ids.npy"

In [ ]:
import numpy as np

# ── salience lookup: id → score ──────────────────────────────────────
# Weights: upvotes, likes, reputation, log-views
W_UP   = 0.35
W_LK   = 0.35
W_REP  = 0.20
W_VIEW = 0.10

def _norm(arr):
    """Min-max normalise, returns values in [0, 1]."""
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)

# Pull fields from your TinyDB (or directly from db_records)
db_records = db.all()   # <- your existing TinyDB handle

upvotes    = _norm(np.array([r.get("upvotes", 0)           for r in db_records], dtype=float))
likes      = _norm(np.array([r.get("likes", 0)             for r in db_records], dtype=float))
reputation = _norm(np.array([r.get("author_reputation", 0) for r in db_records], dtype=float))
views      = _norm(np.log1p(np.array([r.get("views", 0)    for r in db_records], dtype=float)))

salience_raw = W_UP * upvotes + W_LK * likes + W_REP * reputation + W_VIEW * views
salience_arr = _norm(salience_raw)  # re-normalise composite to [0,1]

# Map prompt_id → salience score
salience_map = {r["id"]: salience_arr[i] for i, r in enumerate(db_records)}

In [ ]:
# ── MMR: Maximal Marginal Relevance ─────────────────────────────────
LAM = 1.0   # 1.0 = pure relevance, 0.0 = pure diversity

def mmr_rerank(candidates, embs, lam=LAM, k=N_RETRIEVE):
    selected, remaining = [], list(range(len(candidates)))
    while len(selected) < k and remaining:
        if not selected:
            best = max(remaining, key=lambda i: candidates[i][3])
        else:
            sel_embs = np.array([embs[candidates[i][0]] for i in selected])
            def mmr_score(i):
                e = embs[candidates[i][0]]
                sim_to_selected = np.max(
                    sel_embs @ e / (np.linalg.norm(sel_embs, axis=1) * np.linalg.norm(e) + 1e-9)
                )
                return lam * candidates[i][3] - (1 - lam) * sim_to_selected
            best = max(remaining, key=mmr_score)
        selected.append(best)
        remaining.remove(best)
    return [candidates[i] for i in selected]

In [ ]:
from IPython.display import display, HTML
import numpy as np

ids        = list(np.load(ids_path))
id_to_row  = {pk: i for i, pk in enumerate(ids)}

ALPHA        = 0.7
N_RETRIEVE   = 10
SPOT_QUERIES = list(idx.keys())
pk_meta      = {item["id"]: item for item in dataset}

# ── palette ───────────────────────────────────────────────────────────────────
BG   = "#0f1117"; SURF = "#1a1d27"; SURF2 = "#22263a"
BDR  = "#2e3350"; TXT  = "#e2e6f0"; MUT   = "#7a82a0"
ASP  = "#a86fdf"; GRN  = "#6daa45"; RED   = "#dd6974"; ORG  = "#fdab43"
TEAL = "#4f98a3"; YLW  = "#e8b84b"; AMBER = "#fdab43"; BLUE = "#4a90d9"


def prompt_snippet(pk, max_chars=140):
    item = pk_meta.get(pk, {})
    body = item.get("content", "—")
    snip = body[:max_chars] + ("…" if len(body) > max_chars else "")
    return snip, item.get("title", pk), item.get("category", ""), item.get("difficulty", "")


def _rank_badge(rank, total):
    col = GRN if rank <= 2 else (RED if rank >= total - 1 else MUT)
    return (f'<span style="background:{SURF2};color:{col};font-family:monospace;'
            f'font-size:12px;font-weight:700;padding:2px 8px;border-radius:5px">#{rank}</span>')


def _target_badge(rank, total, not_found=False):
    if not_found:
        return (f'<span style="background:{SURF2};color:{RED};font-family:monospace;'
                f'font-size:11px;font-weight:700;padding:2px 8px;border-radius:5px">'
                f'✗ not in top-{total}</span>')
    col = GRN if rank <= 3 else (YLW if rank <= total // 2 else RED)
    return (f'<span style="background:{SURF2};color:{col};font-family:monospace;'
            f'font-size:11px;font-weight:700;padding:2px 8px;border-radius:5px">'
            f'🎯 #{rank}/{total}</span>')


def _result_row(rank, pk, score, expected_pk, total, highlight_color):
    snip, title, cat, diff = prompt_snippet(pk)
    is_target = pk == expected_pk
    row_bg    = "#162816" if is_target else SURF
    border_l  = f"border-left:3px solid {highlight_color}" if is_target else ""
    tgt_badge = (f' <span style="color:{GRN};font-size:10px;font-weight:700;'
                 f'background:#1e3a1e;padding:1px 5px;border-radius:3px">TARGET</span>'
                 if is_target else "")
    item = pk_meta.get(pk, {})
    ups  = item.get("upvotes", 0)
    lks  = item.get("likes", 0)
    eng  = f'<span style="color:{MUT};font-size:10px">▲{ups} ♥{lks}</span>'
    return f"""
<tr style="background:{row_bg};border-bottom:1px solid {BDR};{border_l}">
  <td style="padding:6px 10px;vertical-align:top;white-space:nowrap">
    {_rank_badge(rank, total)}
  </td>
  <td style="padding:6px 10px;vertical-align:top;max-width:200px">
    <code style="color:{ASP};font-size:11px">{pk}</code>{tgt_badge}<br>
    <span style="color:{TXT};font-weight:600;font-size:12px">{title}</span><br>
    <span style="color:{MUT};font-size:10px">{cat}</span><br>{eng}
  </td>
  <td style="padding:6px 10px;vertical-align:top;font-family:monospace;
      color:{MUT};font-size:11px;max-width:300px;word-break:break-word">{snip}</td>
  <td style="padding:6px 10px;vertical-align:middle;text-align:right;
      font-family:monospace;color:{ASP};font-size:12px;font-weight:700;
      white-space:nowrap">{score:.5f}</td>
</tr>"""


def _separator_row(hidden):
    return (f'<tr style="background:{SURF2}"><td colspan="4" style="padding:4px 10px;'
            f'color:{MUT};font-size:10px;text-align:center">·· {hidden} hidden ··</td></tr>')


# ── normalise hit tuples to (row_index, display_score) regardless of source ──
def _normalise_hits(hits):
    """Accept (ri, sc) from aspace OR (ri, sem, sal, final) from reranker."""
    out = []
    for h in hits:
        if len(h) == 4:          # reranked: use final score
            out.append((h[0], h[3]))
        else:                     # raw aspace: (ri, sc)
            out.append((h[0], h[1]))
    return out


def _build_table(hits, ids, expected_pk, n_retrieve, highlight_color):
    hits    = _normalise_hits(hits)
    top_n   = hits[:n_retrieve]
    total   = len(top_n)

    target_rank = next(
        (r for r, (ri, _) in enumerate(top_n, 1) if ids[ri] == expected_pk), None
    )

    top2    = top_n[:2]
    bottom2 = top_n[max(2, total - 2):]   # FIX: avoid overlapping with top2
    hidden  = max(0, total - 4)

    rows_html = ""
    for rank, (ri, sc) in enumerate(top2, 1):
        rows_html += _result_row(rank, ids[ri], sc, expected_pk, total, highlight_color)
    if hidden > 0:
        rows_html += _separator_row(hidden)
    # FIX: correct start rank for bottom rows
    bottom_start = max(3, total - len(bottom2) + 1)
    for rank, (ri, sc) in enumerate(bottom2, bottom_start):
        rows_html += _result_row(rank, ids[ri], sc, expected_pk, total, highlight_color)

    badge = _target_badge(target_rank, total, not_found=(target_rank is None))
    return rows_html, badge, total, target_rank


# ── column header ─────────────────────────────────────────────────────────────
def _col_header(label, color, badge, total):
    return f"""
<div style="background:{SURF2};padding:8px 12px;border-bottom:2px solid {color}">
  <span style="color:{color};font-weight:700;font-size:13px">{label}</span>
  <span style="float:right">{badge}
    <span style="color:{MUT};font-size:11px;margin-left:8px">{total} results</span>
  </span>
</div>
<table style="width:100%;border-collapse:collapse;background:{SURF}">
  <tr style="background:{SURF2}">
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left;width:50px">Rank</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left;width:200px">Prompt</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left">Preview</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:right;width:80px">Score</th>
  </tr>"""


# ── main render ───────────────────────────────────────────────────────────────
html = (f'<div style="background:{BG};padding:20px;border-radius:14px;'
        f'font-family:system-ui,sans-serif;max-width:1600px">'
        f'<h2 style="color:{TXT};margin:0 0 4px">ArrowSpace Spot-Check — Base vs MMR</h2>'
        f'<p style="color:{MUT};font-size:12px;margin:0 0 18px">'
        f'α={ALPHA} · top-2 and bottom-2 of {N_RETRIEVE} · '
        f'<span style="color:{BLUE}">■ Base</span>  '
        f'<span style="color:{AMBER}">■ MMR (λ={LAM})</span></p>')

wins_mmr = wins_base = ties = 0

for qid in SPOT_QUERIES:
    meta_q      = idx[qid]
    row_i       = meta_q["row_index"]
    expected_pk = meta_q["expected_prompt_id"]
    query_type  = meta_q.get("query_type", "")
    query_text  = meta_q.get("query_text", meta_q.get("query", ""))
    q_vec       = q_embs[row_i].astype(np.float64)

    # Base: raw aspace search — returns (row_idx, score) tuples
    hits_base = aspace_base.search(q_vec, gl_base, ALPHA)

    # MMR: wider candidate pool → salience rerank → MMR diversity filter
    candidates = aspace.search(q_vec, gl_meta, 0.7)
    hits_mmr   = mmr_rerank(candidates, embs, lam=LAM, k=N_RETRIEVE)

    rows_base, badge_base, total_base, rank_base = _build_table(
        hits_base, ids, expected_pk, N_RETRIEVE, BLUE)
    rows_mmr,  badge_mmr,  total_mmr,  rank_mmr  = _build_table(
        hits_mmr,  ids, expected_pk, N_RETRIEVE, AMBER)

    # ── win indicator ──────────────────────────────────────────────────────────
    r_b = rank_base if rank_base else 999
    r_m = rank_mmr  if rank_mmr  else 999
    if r_m < r_b:
        win_label = f'<span style="color:{GRN};font-weight:700">▲ MMR wins (#{r_m} vs #{r_b})</span>'
        wins_mmr += 1
    elif r_b < r_m:
        win_label = f'<span style="color:{BLUE};font-weight:700">▲ BASE wins (#{r_b} vs #{r_m})</span>'
        wins_base += 1
    else:
        win_label = f'<span style="color:{MUT}">= TIE (both #{r_b})</span>'
        ties += 1

    query_block = ""
    if query_text:
        query_block = (f'<div style="background:{BG};border-left:3px solid {TEAL};'
                       f'padding:8px 12px;margin:8px 0 0;font-size:12px;color:{TXT};'
                       f'font-style:italic">'
                       f'<span style="color:{TEAL};font-style:normal;font-weight:600;'
                       f'font-size:10px;display:block;margin-bottom:3px">QUERY</span>'
                       f'{query_text}</div>')

    html += f"""
<div style="border:1px solid {BDR};border-radius:10px;overflow:hidden;margin-bottom:20px">
  <div style="background:{SURF2};padding:10px 14px;display:flex;
      align-items:center;gap:10px;flex-wrap:wrap">
    <span style="font-size:15px;font-weight:700;color:{TXT}">{qid}</span>
    <span style="background:{BG};color:{MUT};font-size:11px;padding:2px 8px;
        border-radius:999px">{query_type}</span>
    <span style="font-size:11px;color:{MUT}">target:
      <code style="color:{ORG};background:{BG};padding:1px 6px;
          border-radius:3px">{expected_pk}</code>
    </span>
    <span style="margin-left:auto;font-size:12px">{win_label}</span>
  </div>
  {query_block}
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:0;
      border-top:1px solid {BDR}">
    <div style="border-right:1px solid {BDR}">
      {_col_header("Base — semantic only", BLUE, badge_base, total_base)}
      {rows_base}
      </table>
    </div>
    <div>
      {_col_header("MMR — diversity reranked", AMBER, badge_mmr, total_mmr)}
      {rows_mmr}
      </table>
    </div>
  </div>
</div>"""

# ── summary scoreboard ─────────────────────────────────────────────────────────
total_q = len(SPOT_QUERIES)
html += f"""
<div style="background:{SURF2};border-radius:10px;padding:16px 20px;
    display:flex;gap:32px;align-items:center;flex-wrap:wrap">
  <span style="color:{TXT};font-weight:700;font-size:14px">Scoreboard</span>
  <span style="color:{AMBER};font-weight:700">MMR wins: {wins_mmr}/{total_q}</span>
  <span style="color:{BLUE};font-weight:700">Base wins: {wins_base}/{total_q}</span>
  <span style="color:{MUT}">Ties: {ties}/{total_q}</span>
</div>"""

html += '</div>'
display(HTML(html))